# CRH-binned comparison across two WRF storms

In [ ]:
from pathlib import Path
import pickle

from matplotlib import colors, ticker, rc
from matplotlib.lines import Line2D
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns
import xarray as xr

DATA_DIR = Path('data')
FIGURE_DIR = Path('figures')
FIGURE_DIR.mkdir(exist_ok=True)

WRF_PATH1 = DATA_DIR / 'binned_2d_ctl_sf_lwprm_10memb_48hrs_71bins_haiyan.pkl'
WRF_PATH2 = DATA_DIR / 'binned_2d_ctl_sf_lwprm_10memb_48hrs_71bins_maria.pkl'

In [ ]:
# Read each native dataset and expose the few fields needed by the common plot.
DISPLAYED_CLASS_INDICES = (1, 4, 5)
XLIM = (.4, .95)


def peak_normalize_displayed_classes(model_tag, values, class_indices=DISPLAYED_CLASS_INDICES):
    """Scale a bin-by-class array by its largest displayed-class value."""
    values = np.asarray(values, dtype=float)
    displayed = values[:, class_indices]
    if not np.any(np.isfinite(displayed)):
        return np.full_like(values, np.nan)
    peak = np.nanmax(displayed)
    if not np.isfinite(peak) or peak <= 0:
        return np.full_like(values, np.nan)
    print(f"Peak {model_tag} displayed-class value: {peak:.3g}")
    return values / peak


def bins_in_displayed_range(x, bounds=XLIM):
    """Select plotted bin coordinates within inclusive bounds."""
    x = np.asarray(x, dtype=float)
    tolerance = 10 * np.finfo(float).eps * max(1, abs(bounds[0]), abs(bounds[1]))
    return (x >= bounds[0] - tolerance) & (x <= bounds[1] + tolerance)


# Read WRF
with WRF_PATH1.open('rb') as file:
    (wrf_bins, wrf_frequency, wrf_lw_theta, wrf_mass_flux, wrf_acre, wrf_rain,
     wrf_class_count, _, _) = pickle.load(file)
with WRF_PATH2.open('rb') as file:
    (wrf_bins2, wrf_frequency2, wrf_lw_theta2, wrf_mass_flux2, wrf_acre2, wrf_rain2,
     wrf_class_count2, _, _) = pickle.load(file)

wrf_pressure = np.arange(1000, 25, -25)
wrf_lw = wrf_lw_theta * (wrf_pressure[np.newaxis, :] / 1000) ** (287 / 1004)
wrf_class_count = np.asarray(wrf_class_count, dtype=float)

wrf_lw2 = wrf_lw_theta2 * (wrf_pressure[np.newaxis, :] / 1000) ** (287 / 1004)
wrf_class_count2 = np.asarray(wrf_class_count2, dtype=float)

base_array = [0.1, 0.5, 1, 5, 10, 50, 100, 500, 1000]
# base_array = np.logspace(-2, 3, num=6)
levels = np.r_[-np.flip(base_array), base_array]

models = [
    dict(
        title='Haiyan', x=np.asarray(wrf_bins[:-1]), pressure=wrf_pressure,
        heating=np.asarray(wrf_lw).T, mass_flux=np.asarray(wrf_mass_flux).T,
        class_occurrence=peak_normalize_displayed_classes('WRF', wrf_class_count),
        sample_count_by_bin=np.asarray(wrf_frequency),
        rain=np.asarray(wrf_rain), acre=np.asarray(wrf_acre),
        levels=levels
    ),
    dict(
        title='Maria', x=np.asarray(wrf_bins2[:-1]), pressure=wrf_pressure,
        heating=np.asarray(wrf_lw2).T, mass_flux=np.asarray(wrf_mass_flux2).T,
        class_occurrence=peak_normalize_displayed_classes('WRF', wrf_class_count2),
        sample_count_by_bin=np.asarray(wrf_frequency2),
        rain=np.asarray(wrf_rain2), acre=np.asarray(wrf_acre2),
        levels=levels
    )
]

In [ ]:
font = {'family': 'sans-serif', 'weight': 'normal', 'size': 10}
rc('font', **font)
sns.set_theme(style='ticks', font_scale=1.2, rc={
    'xtick.bottom': True, 'ytick.left': True,
    'axes.spines.right': False, 'axes.spines.top': False,
})

In [ ]:
# All plotting logic is kept in one cell for quick style edits.
fig, axs = plt.subplots(2, 2, figsize=(11, 5.5), height_ratios=[.65, .35],
                        layout='constrained', dpi=200)

norm = colors.TwoSlopeNorm(vmin=-6, vcenter=0, vmax=3.5)

linewidth = 1.3
images = []
xlim = XLIM
ylim_occurrence = (0, 1)
ylim_rain = (0, 10)
ylim_acre = (-5, 125)

for column, (model, upper, lower) in enumerate(zip(models, axs[0], axs[1])):
    x = model['x']
    pressure = model['pressure']
    x2d = np.broadcast_to(x, np.shape(model['heating']))

    image = upper.pcolormesh(x2d, pressure, model['heating'], cmap='RdBu_r',
                               norm=norm, alpha=.8, shading='auto', zorder=2)
    images.append(image)
    # WRF & SAM use a shared 1-D pressure coordinate; MPAS pressure varies by CRH bin.
    contour_x = x if np.ndim(pressure) == 1 else x2d
    contour = upper.contour(contour_x, pressure, model['mass_flux'], levels=model['levels'],
                            colors='black', zorder=3, linewidths=0.7)
    upper.clabel(contour, contour.levels, inline=True, fontsize=11, fmt='%.3g')
    # Add lettering to each panel
    letter = '(' + chr(ord('a') + column) + ') '
    upper.set_title(letter + model['title'])
    upper.set_xlim(*xlim)
    upper.set_ylim(100, np.nanmax(pressure))
    upper.invert_yaxis()
    upper.set_yscale('log')
    upper.set_xticks([])
    upper.yaxis.set_major_formatter(ticker.ScalarFormatter())
    upper.yaxis.set_minor_formatter(ticker.ScalarFormatter())
    upper.set_yticks([1000, 800, 600, 500, 400, 300, 200, 150, 100], minor=True)
    sns.despine(offset=10, ax=upper, bottom=True)

    for class_index, label, linestyle in [(1, 'Deep', '-'), (4, 'Strat', '--'), (5, 'Anvil', ':')]:
        lower.plot(x, model['class_occurrence'][:, class_index], color='k', linestyle=linestyle,
                   linewidth=linewidth, label=label)
    lower.set_xlim(*xlim)
    lower.set_ylim(*ylim_occurrence)
    if len(models) == 2 and column == 1:
        upper.invert_xaxis()
        lower.invert_xaxis()

    lower.set_xlabel('Column saturation fraction')
    sns.despine(offset=10, ax=lower, left=False, bottom=False, right=True, top=True)

    rain_axis = lower.twinx()
    rain_axis.plot(x, model['rain'], '-r', linewidth=linewidth)
    rain_axis.spines['right'].set_position(('outward', 5))
    rain_axis.set_ylim(*ylim_rain)
    rain_axis.spines['right'].set_color('r')
    rain_axis.tick_params(axis='y', which='both', colors='r')

    acre_axis = lower.twinx()
    acre_axis.plot(x, model['acre'], '--g', linewidth=linewidth)
    acre_axis.spines['right'].set_position(('outward', 65))
    acre_axis.spines['right'].set_color('g')
    acre_axis.tick_params(axis='y', which='both', colors='g')
    acre_axis.set_ylim(*ylim_acre)
    for axis in (rain_axis, acre_axis):
        axis.spines['left'].set_visible(False)
        axis.tick_params(axis='y', which='both', left=False, labelleft=False)

    # The primary y axes are shared visually; show them only on the first column.
    if column == 0:
        upper.set_ylabel('Pressure [hPa]')
        lower.set_ylabel('Peak-normalized\noccurrence')
    else:
        upper.tick_params(axis='y', which='both', left=False, labelleft=False)
        upper.spines['left'].set_visible(False)
        lower.tick_params(axis='y', which='both', left=False, labelleft=False)
        lower.spines['left'].set_visible(False)

    if column == len(models) - 1:
        rain_axis.set_ylabel(r'$P$ [mm/hr]', color='r')
        acre_axis.set_ylabel('LW-ACRE [W/m$^2$]', color='g')
        rain_axis.spines['right'].set_visible(True)
        acre_axis.spines['right'].set_visible(True)
    else:
        for axis in (rain_axis, acre_axis):
            axis.tick_params(axis='y', which='both', right=False, labelright=False)
            axis.spines['right'].set_visible(False)
    rain_axis.spines['bottom'].set_visible(False)
    acre_axis.spines['bottom'].set_visible(False)

fig.colorbar(image, ax=axs[0,1], shrink=.8, ticks=ticker.AutoLocator(),
             label='K/d', extend='both', pad = -0.30,
             location='right')

handles = [
    Line2D([0], [0], color='k', linestyle='-', label='Deep'),
    Line2D([0], [0], color='k', linestyle='--', label='Strat'),
    Line2D([0], [0], color='k', linestyle=':', label='Anvil'),
    Line2D([0], [0], color='r', linestyle='-', label=r'$P$'),
    Line2D([0], [0], color='g', linestyle='--', label='LW-ACRE'),
]
axs[1, 0].legend(handles=handles, frameon=False, loc='upper left', fontsize=10)

figname = 'crh_binned_two_storms.pdf'
figure_path = FIGURE_DIR / figname
fig.savefig(figure_path, bbox_inches='tight')
plt.show()